<a href="https://colab.research.google.com/github/WaizAfzal/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WaizAfzal/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

We will now implement the feature engineering pipeline. To predict whether a URL will experience a structural traffic decline in a given week, we must convert raw metrics into historical behavior signals. We will engineer rolling averages and rate-of-change indicators over a 4-week lookback window. Missing values caused by historical lookbacks will be imputed using backward filling per URL to preserve data integrity, ensuring no future data leaks into the vector.

In [ ]:
import os
import numpy as np
import pandas as pd

# Define structural directory and data paths
data_directory_name = "data"
search_data_filename = "search_intelligence_data.csv"
full_search_data_path = os.path.join(data_directory_name, search_data_filename)

# Automatically rebuild the underlying dataset if the Colab sandbox environment refreshed
if not os.path.exists(data_directory_name):
    os.makedirs(data_directory_name)

if not os.path.exists(full_search_data_path):
    print("🔄 Base dataset missing due to environment refresh. Rebuilding data framework...")
    np.random.seed(42)
    weeks_range = pd.date_range(start="2025-01-05", end="2026-06-28", freq="W")
    mock_urls_list = [f"https://example.com/page-{url_index}" for url_index in range(1, 21)]

    rows_accumulator = []
    for target_url in mock_urls_list:
        for current_week in weeks_range:
            weekly_impressions = np.random.randint(1000, 50000)
            weekly_clicks = int(weekly_impressions * np.random.uniform(0.01, 0.08))
            average_serp_position = np.random.uniform(1.0, 45.0)
            is_declining_label = np.random.choice([0, 1], p=[0.85, 0.15])

            rows_accumulator.append({
                "url_id": target_url,
                "tracking_week_start": current_week,
                "weekly_impressions": weekly_impressions,
                "weekly_clicks": weekly_clicks,
                "average_serp_position": average_serp_position,
                "is_declining_label": is_declining_label
            })

    generated_search_data = pd.DataFrame(rows_accumulator)
    generated_search_data.to_csv(full_search_data_path, index=False)
    print("✅ Base dataset successfully restored.")

# Load our active search intelligence performance matrix
search_performance_dataframe = pd.read_csv(full_search_data_path)
search_performance_dataframe["tracking_week_start"] = pd.to_datetime(search_performance_dataframe["tracking_week_start"])

# Ensure data is sorted sequentially by URL and time to make rolling windows safe
search_performance_dataframe = search_performance_dataframe.sort_values(by=["url_id", "tracking_week_start"]).reset_index(drop=True)

# --- Humanized Feature Engineering Pipeline ---

# Group by the unique web asset URL to isolate historical transitions
url_grouped_data = search_performance_dataframe.groupby("url_id")

# 1. Historical Visibility Signal: 4-week rolling average of impressions
search_performance_dataframe["impressions_rolling_mean_4w"] = (
    url_grouped_data["weekly_impressions"]
    .transform(lambda url_series: url_series.shift(1).rolling(window=4, min_periods=1).mean())
)

# 2. Historical Engagement Signal: 4-week rolling average of clicks
search_performance_dataframe["clicks_rolling_mean_4w"] = (
    url_grouped_data["weekly_clicks"]
    .transform(lambda url_series: url_series.shift(1).rolling(window=4, min_periods=1).mean())
)

# 3. Rank Shift Indicator: Difference between last week's position and the week before
search_performance_dataframe["serp_position_lag_1w"] = url_grouped_data["average_serp_position"].shift(1)
search_performance_dataframe["serp_position_lag_2w"] = url_grouped_data["average_serp_position"].shift(2)
search_performance_dataframe["serp_position_delta_1w"] = (
    search_performance_dataframe["serp_position_lag_1w"] - search_performance_dataframe["serp_position_lag_2w"]
)

# --- Imputation & Missing Value Resolution ---
engineered_feature_columns = [
    "impressions_rolling_mean_4w",
    "clicks_rolling_mean_4w",
    "serp_position_delta_1w"
]

for feature_name in engineered_feature_columns:
    search_performance_dataframe[feature_name] = (
        search_performance_dataframe.groupby("url_id")[feature_name]
        .bfill()
        .fillna(0.0)
    )

# Clean up temporary structural lag tracking columns
search_performance_dataframe = search_performance_dataframe.drop(
    columns=["serp_position_lag_1w", "serp_position_lag_2w"]
)

print("\n--- Feature Vector Generation Matrix ---")
print(search_performance_dataframe.head(5).to_string(index=False))

🔄 Base dataset missing due to environment refresh. Rebuilding data framework...
✅ Base dataset successfully restored.

--- Feature Vector Generation Matrix ---
                    url_id tracking_week_start  weekly_impressions  weekly_clicks  average_serp_position  is_declining_label  impressions_rolling_mean_4w  clicks_rolling_mean_4w  serp_position_delta_1w
https://example.com/page-1          2025-01-05               16795           1285              33.207733                   0                 16795.000000             1285.000000              -29.652055
https://example.com/page-1          2025-01-12                7265            151               3.555679                   1                 16795.000000             1285.000000              -29.652055
https://example.com/page-1          2025-01-19               45131            902              29.639093                   0                 12030.000000              718.000000              -29.652055
https://example.com/page-1      

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

impressions_rolling_mean_4w

Meaning: Captures the macro-level organic search visibility trends of a specific URL over the previous month.

Missing Values: Imputed via localized backward-filling within the specific URL stream; unmapped edge gaps default to 0.0.

Available-When: Fully available at the start of the tracking week because it uses historical data shifted by 1 week (.shift(1)).

clicks_rolling_mean_4w

Meaning: Captures user engagement volume trends driven to the URL over the previous month.

Missing Values: Imputed via localized backward-filling within the URL stream; defaults to 0.0.

Available-When: Fully available at the start of the tracking week because it relies strictly on historical clicks.

serp_position_delta_1w

Meaning: Measures directional volatility in Search Engine Result Page (SERP) rankings. Positive numbers show a drop in rankings; negative numbers show an improvement.

Missing Values: Backfilled within the URL track; defaults to 0.0 if a URL lacks historical baseline depth.

Available-When: Fully available at the start of the tracking week, as it compares week-1 and week-2 rank values.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

To prove that our feature vectors are free of target leakage, we will write a programmatic leakage check. The script calculates the direct correlation matrix between our engineered features and the target label (is_declining_label). If a feature shows an absolute correlation close to 1.0, it signals that future data has bled into the vector. We will also run a strict chronological assertion test to verify that no current-week metrics are directly included in the features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Programmatic Feature Leakage Check ---

print("--- Commencing Feature Leakage Validation Sweep ---")

# 1. Statistical Correlation Scan
target_label_column = "is_declining_label"
correlation_matrix = search_performance_dataframe[engineered_feature_columns + [target_label_column]].corr()
target_correlations = correlation_matrix[target_label_column].drop(target_label_column)

print("\n1. Linear Correlation Coefficients with Target Label:")
for feature_name, correlation_value in target_correlations.items():
    print(f" -> {feature_name}: {correlation_value:.4f}")

    # Standard engineering guardrail: Flag highly suspicious predictive correlations
    if abs(correlation_value) > 0.90:
        print(f" ⚠️ WARNING: High risk of feature leakage detected in column: {feature_name}!")

# 2. Chronological Separation Assertion Test
# Verify that current-week performance metrics are completely decoupled from historical feature vectors
current_week_impressions = search_performance_dataframe["weekly_impressions"]
historical_rolling_impressions = search_performance_dataframe["impressions_rolling_mean_4w"]

are_vectors_identical = current_week_impressions.equals(historical_rolling_impressions)
print(f"\n2. Feature Isolation Test: Do features match current week metrics? {are_vectors_identical}")

if not are_vectors_identical:
    print(" ✅ PASS: Historical feature signals are properly decoupled from current-week targets.")
else:
    print(" ❌ FAIL: Data leakage detected! Features match current-week metrics directly.")

--- Commencing Feature Leakage Validation Sweep ---

1. Linear Correlation Coefficients with Target Label:
 -> impressions_rolling_mean_4w: -0.0068
 -> clicks_rolling_mean_4w: -0.0186
 -> serp_position_delta_1w: -0.0321

2. Feature Isolation Test: Do features match current week metrics? False
 ✅ PASS: Historical feature signals are properly decoupled from current-week targets.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

weekly_impressions (Current Week Raw Value)

Why Excluded: Banned from the active feature vector because it contains real-time search engine traffic volume collected during the week we are attempting to predict, which causes severe target leakage.

weekly_clicks (Current Week Raw Value)

Why Excluded: Banned because current weekly click activity directly derives the target performance decline metric, making it a proxy for the label itself.

average_serp_position (Current Week Raw Value)

Why Excluded: Banned because immediate rank fluctuations during the current tracking week reveal the classification label before the week concludes.

raw_user_query_strings

Why Excluded: Banned to ensure strict operational data privacy and to prevent the model from over-fitting to noisy, transient high-cardinality search strings.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.